In [1]:
import os


In [7]:
inputpath = "../data/"
outputpath = "../data/bio/"

In [8]:
from spacy.lang.en import English
nlp = English()
sentencizer = nlp.create_pipe("sentencizer")
nlp.add_pipe(sentencizer)

ValueError: [E966] `nlp.add_pipe` now takes the string name of the registered component factory, not a callable component. Expected string, but got <spacy.pipeline.sentencizer.Sentencizer object at 0x000001697EEA9880> (name: 'None').

- If you created your component with `nlp.create_pipe('name')`: remove nlp.create_pipe and call `nlp.add_pipe('name')` instead.

- If you passed in a component like `TextCategorizer()`: call `nlp.add_pipe` with the string name instead, e.g. `nlp.add_pipe('textcat')`.

- If you're using a custom component: Add the decorator `@Language.component` (for function components) or `@Language.factory` (for class components / factories) to your custom component and assign it a name, e.g. `@Language.component('your_name')`. You can then run `nlp.add_pipe('your_name')` to add it to the pipeline.

In [9]:
inputfiles = set()
for f in os.listdir(inputpath):
    if f.endswith('.ann'):
        inputfiles.add(f.split('.')[0].split('_')[0])
len(inputfiles)

FileNotFoundError: [WinError 3] Das System kann den angegebenen Pfad nicht finden: '../data/'

In [10]:
select_types = ['Condition', 'Value', 'Drug', 'Procedure', 'Measurement', 'Temporal', \
                'Observation', 'Person', 'Mood', 'Device', 'Pregnancy_considerations']

In [11]:
# convert Brat format into BIO format
# function for getting entity annotations from the annotation file
def get_annotation_entities(ann_file, select_types=None):
    entities = []
    with open(ann_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith('T'):
                term = line.strip().split('\t')[1].split()
                if (select_types != None) and (term[0] not in select_types): continue
                if int(term[-1]) <= int(term[1]): continue
                entities.append((int(term[1]), int(term[-1]), term[0]))
    return sorted(entities, key=lambda x: (x[0], x[1]))

# function for handling overlap by keeping the entity with largest text span
def remove_overlap_entities(sorted_entities):
    keep_entities = []
    for idx, entity in enumerate(sorted_entities):
        if idx == 0:
            keep_entities.append(entity)
            last_keep = entity
            continue
        if entity[0] < last_keep[1]:
            if entity[1]-entity[0] > last_keep[1]-last_keep[0]:
                last_keep = entity
                keep_entities[-1] = last_keep
        elif entity[0] == last_keep[1]:
            last_keep = (last_keep[0], entity[1], last_keep[-1])
            keep_entities[-1] = last_keep
        else:
            last_keep = entity
            keep_entities.append(entity)
    return keep_entities

# inverse index of entity annotations
def entity_dictionary(keep_entities, txt_file):
    f_ann = {}
    with open(txt_file, "r", encoding="utf-8") as f:
        text = f.readlines()
        if file in ['NCT02348918_exc', 'NCT02348918_inc', 'NCT01735955_exc']:
            text = ' '.join([i.strip() for i in text])
        else:
            text = '  '.join([i.strip() for i in text])
    for entity in keep_entities:
        entity_text = text[entity[0]:entity[1]]
        doc = nlp(entity_text)
        token_starts = [(i, doc[i:].start_char) for i in range(len(doc))]
        term_type = entity[-1]
        term_offset = entity[0]
        for i, token in enumerate(doc):
            ann_offset = token_starts[i][1]+term_offset
            if ann_offset not in f_ann:
                f_ann[ann_offset] = [i, token.text, term_type]
    return f_ann

# Brat -> BIO format conversion
for infile in inputfiles:
    for t in ["exc", "inc"]:
        file = f"{infile}_{t}"
        ann_file = f"{inputpath}/{file}.ann"
        txt_file = f"{inputpath}/{file}.txt"
        out_file = f"{outputpath}/{file}.bio.txt"
        sorted_entities = get_annotation_entities(ann_file, select_types)
        keep_entities = remove_overlap_entities(sorted_entities)
        f_ann = entity_dictionary(keep_entities, txt_file)
        with open(out_file, "w", encoding="utf-8") as f_out:
            with open(txt_file, "r", encoding="utf-8") as f:
                sent_offset = 0
                for line in f:
                    # print(line.strip())
                    if '⁄' in line:
                        # print(txt_file)
                        line = line.replace('⁄', '/') # replace non unicode characters
                    doc = nlp(line.strip())
                    token_starts = [(i, doc[i:].start_char) for i in range(len(doc))]
                    for token in doc:
                        token_sent_offset = token_starts[token.i][1]
                        token_doc_offset = token_starts[token.i][1]+sent_offset
                        if token_doc_offset in f_ann:
                            if f_ann[token_doc_offset][0] == 0:
                                label = f"B-{f_ann[token_doc_offset][2]}"
                            else:
                                label = f"I-{f_ann[token_doc_offset][2]}"
                        else:
                            label = f"O"
                        # print(token.text, token_sent_offset, token_sent_offset+len(token.text), token_doc_offset, token_doc_offset+len(token.text), label)
                        f_out.write(f"{token.text} {token_sent_offset} {token_sent_offset+len(token.text)} {token_doc_offset} {token_doc_offset+len(token.text)} {label}\n")
                    # print('\n')
                    f_out.write('\n')
                    if file in ['NCT02348918_exc', 'NCT02348918_inc', 'NCT01735955_exc']: # 3 trials with inconsistent offsets
                        sent_offset += (len(line.strip())+1)
                    else:
                        sent_offset += (len(line.strip())+2)

In [12]:
out_file = f"data/bio/chia_ner.tsv"
with open(out_file, "w", encoding="utf-8") as f_out:
    for infile in inputfiles:
        for t in ["exc", "inc"]:
            file = f"{infile}_{t}"
            ann_file = f"{inputpath}/{file}.ann"
            txt_file = f"{inputpath}/{file}.txt"
            sorted_entities = get_annotation_entities(ann_file, select_types)
            keep_entities = remove_overlap_entities(sorted_entities)
            with open(txt_file, "r", encoding="utf-8") as f:
                sent_offset = 0
                for line in f:
                    # print(line.strip())
                    if '⁄' in line: line = line.replace('⁄', '/')
                    sent_end = sent_offset + len(line)
                    sent_ents = []
                    for ent in keep_entities:
                        if ent[0] < sent_offset or ent[1] < sent_offset: continue
                        if ent[0] >= sent_end or ent[1] > sent_offset+len(line.strip()): break
                        ent_start = ent[0]-sent_offset+1
                        ent_end = ent[1]-sent_offset+1
                        sent_ents.append(f"{ent_start}:{ent_end}:{ent[2].lower()}")
                    if sent_ents == []:
                        if file in ['NCT02348918_exc', 'NCT02348918_inc', 'NCT01735955_exc']:
                            sent_offset += (len(line.strip())+1)
                        else:
                            sent_offset += (len(line.strip())+2)
                        continue
                    # print(f"{file}\t{','.join(sent_ents)}\t{line.strip()}")
                    f_out.write(f"{file}\t{','.join(sent_ents)}\t{line.strip()}")
                    # print('\n')
                    f_out.write('\n')
                    if file in ['NCT02348918_exc', 'NCT02348918_inc', 'NCT01735955_exc']:
                        sent_offset += (len(line.strip())+1)
                    else:
                        sent_offset += (len(line.strip())+2)